# 03. Моделирование: WOE-логрегрессия, калибровка PD

ТЗ §2.3:
- **baseline в первый день**: логистическая регрессия на WOE-признаках (интерпретируемая, переводится в скор-карту);
- основная модель: интерпретируемая логистическая регрессия (скор-карта);
- разбиение train / valid / test по компаниям; valid — подбор гиперпараметров и калибровка, **test — только финальная оценка**;
- калибровка вероятностей (Platt / isotonic), чтобы `pd` была настоящей вероятностью;
- артефакт сериализован и версионирован.

## 1. Модули: WOE-кодировщик, обёртки модели, метрики

In [1]:
%%writefile ../risk-scoring/riskscore/woe.py
"""WOE-кодирование с монотонным биннингом (основа скор-карты).

WOE здесь = ln( доля плохих в корзине / доля хороших в корзине ):
положительный WOE -> корзина рискованнее среднего. Пропуск (NaN) - отдельная корзина.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

OTHER, MISSING = "__OTHER__", "__MISSING__"


class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, num_cols=(), cat_cols=(), directions=None, max_bins=8, min_bin_frac=0.05,
                 min_bin_count=30, n_init_bins=20, alpha=0.5, min_missing=30):
        self.num_cols, self.cat_cols, self.directions = list(num_cols), list(cat_cols), directions
        self.max_bins, self.min_bin_frac, self.min_bin_count = max_bins, min_bin_frac, min_bin_count
        self.n_init_bins, self.alpha, self.min_missing = n_init_bins, alpha, min_missing

    # ------------------------------------------------------------------ подбор корзин
    @staticmethod
    def _stats(b, y, k):
        return np.bincount(b, minlength=k), np.bincount(b, weights=y, minlength=k)

    def _edges_num(self, v, y, direction):
        uniq = np.unique(v)
        discrete = len(uniq) <= 6
        if discrete:
            edges = list((uniq[:-1] + uniq[1:]) / 2)
            min_n = self.min_bin_count
        else:
            edges = list(np.unique(np.quantile(v, np.linspace(0, 1, self.n_init_bins + 1)[1:-1])))
            min_n = max(self.min_bin_frac * len(v), self.min_bin_count)
        while edges:
            b = np.searchsorted(edges, v, side="left")
            n, bad = self._stats(b, y, len(edges) + 1)
            rate = bad / np.maximum(n, 1)
            small = np.where(n < min_n)[0]
            if small.size:                                   # 1) слишком маленькая корзина -> слить с ближайшей по доле плохих
                i = small[np.argmin(n[small])]
                if i == 0: j = 0
                elif i == len(n) - 1: j = len(n) - 2
                else: j = i - 1 if abs(rate[i] - rate[i - 1]) <= abs(rate[i] - rate[i + 1]) else i
                edges.pop(j); continue
            if direction != 0:                               # 2) монотонность (ожидаемый знак влияния на риск)
                viol = np.where(direction * np.diff(rate) < 0)[0]
                if viol.size:
                    edges.pop(viol[0]); continue
            if len(n) > self.max_bins:                       # 3) ограничение числа корзин
                edges.pop(int(np.argmin(np.abs(np.diff(rate))))); continue
            break
        return np.array(edges, float)

    def _woe_from_counts(self, n, bad, tot_n, tot_bad):
        k = len(n); a = self.alpha
        good, tot_good = n - bad, tot_n - tot_bad
        db = (bad + a) / (tot_bad + a * k)
        dg = (good + a) / (tot_good + a * k)
        woe = np.log(db / dg)
        return woe, (db - dg) * woe

    def fit(self, X: pd.DataFrame, y):
        y = np.asarray(y, float)
        tot_n, tot_bad = len(y), y.sum()
        dirs = self.directions or {}
        self.bins_ = {}
        for c in self.num_cols:
            x = pd.to_numeric(X[c], errors="coerce").to_numpy(float)
            m = ~np.isnan(x)
            edges = self._edges_num(x[m], y[m], dirs.get(c, 0)) if m.sum() > 0 else np.array([])
            b = np.searchsorted(edges, x[m], side="left")
            n, bad = self._stats(b, y[m], len(edges) + 1)
            labels = [self._label(edges, i) for i in range(len(edges) + 1)]
            has_miss = (~m).sum() >= self.min_missing
            if has_miss:
                n = np.append(n, (~m).sum()); bad = np.append(bad, y[~m].sum()); labels.append(MISSING)
            woe, iv = self._woe_from_counts(n, bad, tot_n, tot_bad)
            self.bins_[c] = dict(kind="num", edges=edges, woe=woe[:len(edges) + 1], miss_woe=float(woe[-1]) if has_miss else 0.0,
                                 table=pd.DataFrame(dict(bin=labels, n=n.astype(int), bad=bad.astype(int), bad_rate=bad / np.maximum(n, 1), woe=woe, iv=iv)))
        for c in self.cat_cols:
            x = X[c].astype(object).where(X[c].notna(), MISSING).astype(str).to_numpy()
            cats, cnt = np.unique(x, return_counts=True)
            keep = [k for k, v in zip(cats, cnt) if v >= max(self.min_bin_frac * len(x) * .5, self.min_bin_count) or k == MISSING and v >= self.min_missing]
            lab = np.where(np.isin(x, keep), x, OTHER)
            uc = list(np.unique(lab))
            n = np.array([(lab == k).sum() for k in uc]); bad = np.array([y[lab == k].sum() for k in uc])
            woe, iv = self._woe_from_counts(n, bad, tot_n, tot_bad)
            self.bins_[c] = dict(kind="cat", mapping=dict(zip(uc, woe)), has_other=OTHER in uc,
                                 table=pd.DataFrame(dict(bin=uc, n=n.astype(int), bad=bad.astype(int), bad_rate=bad / np.maximum(n, 1), woe=woe, iv=iv)))
        self.iv_ = pd.Series({c: b["table"]["iv"].sum() for c, b in self.bins_.items()}).sort_values(ascending=False)
        return self

    @staticmethod
    def _label(edges, i):
        lo = "-inf" if i == 0 else f"{edges[i - 1]:.4g}"
        hi = "+inf" if i == len(edges) else f"{edges[i]:.4g}"
        return f"({lo}; {hi}]"

    # ------------------------------------------------------------------ применение
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        out = {}
        for c, b in self.bins_.items():
            if b["kind"] == "num":
                x = pd.to_numeric(X[c], errors="coerce").to_numpy(float)
                w = np.full(len(x), b["miss_woe"])
                m = ~np.isnan(x)
                if len(b["woe"]):
                    w[m] = b["woe"][np.searchsorted(b["edges"], x[m], side="left")]
                out[c] = w
            else:
                x = X[c].astype(object).where(X[c].notna(), MISSING).astype(str)
                mp = b["mapping"]
                out[c] = x.map(lambda v: mp.get(v, mp.get(OTHER, 0.0))).to_numpy(float)
        return pd.DataFrame(out, index=X.index)

    def bin_index(self, X: pd.DataFrame, col: str) -> np.ndarray:
        """Индекс корзины (для скор-карты)."""
        b = self.bins_[col]; x = X[col]
        if b["kind"] == "num":
            v = pd.to_numeric(x, errors="coerce").to_numpy(float)
            idx = np.full(len(v), len(b["woe"]))               # последняя строка таблицы = MISSING
            m = ~np.isnan(v)
            idx[m] = np.searchsorted(b["edges"], v[m], side="left")
            return idx
        labels = list(b["table"]["bin"])
        x = x.astype(object).where(x.notna(), MISSING).astype(str)
        return x.map(lambda v: labels.index(v) if v in labels else (labels.index(OTHER) if OTHER in labels else 0)).to_numpy()

Overwriting ../risk-scoring/riskscore/woe.py


In [2]:
%%writefile ../risk-scoring/riskscore/metrics.py
"""Метрики качества скоринговой модели: различающая способность, точность вероятностей, стабильность."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve


def gini(y, p) -> float:
    return 2 * roc_auc_score(y, p) - 1


def ks(y, p) -> float:
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))


def brier(y, p) -> float:
    return float(brier_score_loss(y, p))


def calibration_table(y, p, n_bins=10) -> pd.DataFrame:
    """Квантильные корзины по прогнозу: средний прогноз vs фактическая доля дефолтов."""
    d = pd.DataFrame({"y": np.asarray(y, float), "p": np.asarray(p, float)})
    d["bin"] = pd.qcut(d.p.rank(method="first"), n_bins, labels=False)
    t = d.groupby("bin").agg(n=("y", "size"), mean_pd=("p", "mean"), observed=("y", "mean")).reset_index(drop=True)
    return t


def ece(y, p, n_bins=10) -> float:
    t = calibration_table(y, p, n_bins)
    return float((t.n * (t.mean_pd - t.observed).abs()).sum() / t.n.sum())


def psi(expected, actual, n_bins=10, eps=1e-4) -> float:
    """Population Stability Index по квантильным корзинам распределения `expected`."""
    e, a = np.asarray(expected, float), np.asarray(actual, float)
    edges = np.unique(np.quantile(e, np.linspace(0, 1, n_bins + 1)[1:-1]))
    pe = np.bincount(np.searchsorted(edges, e, side="left"), minlength=len(edges) + 1) / len(e)
    pa = np.bincount(np.searchsorted(edges, a, side="left"), minlength=len(edges) + 1) / len(a)
    pe, pa = np.clip(pe, eps, None), np.clip(pa, eps, None)
    return float(np.sum((pa - pe) * np.log(pa / pe)))


def summary(y, p, name="") -> dict:
    return dict(name=name, n=int(len(y)), default_rate=float(np.mean(y)), auc=float(roc_auc_score(y, p)),
                gini=gini(y, p), ks=ks(y, p), brier=brier(y, p), ece=ece(y, p), mean_pd=float(np.mean(p)))

Overwriting ../risk-scoring/riskscore/metrics.py


In [3]:
%%writefile ../risk-scoring/riskscore/models.py
"""Обёртки: селектор колонок, калиброванная модель PD (единый сериализуемый артефакт)."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

PD_MIN, PD_MAX = 0.002, 0.95        # границы прогноза: PD не бывает ровно 0 или 1


class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns=()):
        self.columns = list(columns)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.columns]


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


class CalibratedPD:
    """pipeline (сырая заявка -> необработанный скор) + калибратор (Platt/isotonic) -> PD.

    Детерминирован: никакого случайного состояния при инференсе.
    """

    def __init__(self, pipeline, calibrator=None, method="none", version="v1", meta=None):
        self.pipeline, self.calibrator, self.method = pipeline, calibrator, method
        self.version, self.meta = version, meta or {}

    def raw_score(self, raw: pd.DataFrame) -> np.ndarray:
        return self.pipeline.predict_proba(raw)[:, 1]

    def calibrate(self, p_raw: np.ndarray) -> np.ndarray:
        if self.method == "platt":
            p = self.calibrator.predict_proba(_logit(p_raw).reshape(-1, 1))[:, 1]
        elif self.method == "isotonic":
            p = self.calibrator.predict(p_raw)
        else:
            p = p_raw
        return np.clip(p, PD_MIN, PD_MAX)

    def predict_pd(self, raw: pd.DataFrame) -> np.ndarray:
        return self.calibrate(self.raw_score(raw))

Overwriting ../risk-scoring/riskscore/models.py


## 2. Данные, разбиение, признаки

In [4]:
import sys, json, time, warnings, joblib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "risk-scoring")); sys.path.insert(0, str(ROOT))
from riskscore.features import (compute_features, QuantileClipper, FEATURE_META, FEATURE_COLUMNS, NUM_FEATURES, CAT_FEATURES, BIN_FEATURES)
from riskscore.woe import WOEEncoder
from riskscore.models import ColumnSelector, CalibratedPD
from riskscore import metrics as M
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
IMG = ROOT / "docs/ml/img"; MODELS = ROOT / "risk-scoring/models"
STR = {"inn": str, "ogrn": str, "okved": str, "region_code": str}
raw = pd.read_csv(ROOT / "data/generated/applications_raw.csv", dtype=STR)
split = pd.read_csv(ROOT / "data/generated/split.csv")
assert (split.company_id == raw.company_id).all()
y = raw.default_flag.astype(int).to_numpy()
tr, va, te = (np.where(split.split == s)[0] for s in ("train", "valid", "test"))
print({s: len(i) for s, i in zip(("train", "valid", "test"), (tr, va, te))})
X_all = compute_features(raw)
clip = QuantileClipper().fit(X_all.iloc[tr])
Xc = clip.transform(X_all)

{'train': 7200, 'valid': 2400, 'test': 2400}


## 3. WOE-биннинг на train: Information Value и монотонность
Для признаков с ожидаемым направлением влияния (`+`/`−` в словаре) корзины строятся **монотонными** — это защита от переобучения и требование интерпретируемости.

In [5]:
dirs = {k: {"+": 1, "-": -1, "~": 0}[m["direction"]] for k, m in FEATURE_META.items()}
woe = WOEEncoder(num_cols=NUM_FEATURES + BIN_FEATURES, cat_cols=CAT_FEATURES, directions=dirs).fit(Xc.iloc[tr], y[tr])
iv = woe.iv_
print("IV (топ-20):"); print(iv.head(20).round(3).to_string())
print("\nПравило: IV <0.02 — бесполезный, 0.02–0.1 слабый, 0.1–0.3 средний, 0.3–0.5 сильный, >0.5 подозрительный")
print("признаков с IV ≥ 0.02:", int((iv >= 0.02).sum()), "из", len(iv), "| с IV > 0.5:", int((iv > 0.5).sum()))
print("\nпример корзин — payment_burden:"); print(woe.bins_["payment_burden"]["table"].round(3).to_string(index=False))
print("\nпример корзин — overdue_90_cnt (пропуск = нет КИ):"); print(woe.bins_["overdue_90_cnt"]["table"].round(3).to_string(index=False))

IV (топ-20):
liabilities_to_ebitda         1.255
net_margin                    1.225
roa                           1.201
ebitda_margin                 1.162
payment_burden                1.108
dscr                          1.101
loss_quarters                 1.058
equity_ratio                  0.981
existing_payment_to_ebitda    0.920
new_payment_to_ebitda         0.904
negative_equity               0.516
gross_margin                  0.402
revenue_growth_yoy            0.362
liabilities_to_revenue        0.337
revenue_trend                 0.334
balance_to_turnover           0.324
inquiries_12m                 0.293
overdue_30_cnt                0.277
industry                      0.266
inquiries_3m                  0.263

Правило: IV <0.02 — бесполезный, 0.02–0.1 слабый, 0.1–0.3 средний, 0.3–0.5 сильный, >0.5 подозрительный
признаков с IV ≥ 0.02: 43 из 64 | с IV > 0.5: 11

пример корзин — payment_burden:
             bin    n  bad  bad_rate    woe    iv
  (-inf; 0.3667] 2160   67    

## 4. Отбор признаков для скор-карты
1. IV ≥ 0.02; 2. убираем сильно коррелированные (|ρ| > 0.75 между WOE, оставляем с большим IV);
3. после обучения убираем признаки с «неправильным» знаком коэффициента (при WOE = ln(плохие/хорошие) все коэффициенты должны быть > 0);
4. исключение: `payment_burden` (долговая нагрузка с учётом суммы и срока заявки) остаётся в модели всегда — иначе PD не реагировала бы на запрошенные условия.

In [6]:
W_tr = woe.transform(Xc.iloc[tr])
cand = [c for c in iv.index if iv[c] >= 0.02]
# доменное требование: скор обязан реагировать на параметры заявки (сумма/срок -> платёж), поэтому совокупная нагрузка включается принудительно
MUST = ["payment_burden"]
kept = []
for c in MUST + [c for c in cand if c not in MUST]:      # cand отсортирован по убыванию IV; обязательные признаки идут первыми
    if all(abs(np.corrcoef(W_tr[c], W_tr[k])[0, 1]) < 0.75 for k in kept):
        kept.append(c)
print(f"после IV-фильтра: {len(cand)}, после фильтра корреляций: {len(kept)}")
dropped_corr = sorted(set(cand) - set(kept)); print("отброшены по корреляции:", dropped_corr)

def fit_lr(cols, C):
    lr = LogisticRegression(C=C, max_iter=2000, solver="lbfgs")
    lr.fit(W_tr[cols], y[tr]); return lr
cols = list(kept); C0 = 0.3
while True:
    lr = fit_lr(cols, C0)
    neg = [(c, b) for c, b in zip(cols, lr.coef_[0]) if b <= 0 and c not in MUST]
    if not neg: break
    worst = min(neg, key=lambda t: t[1])[0]; cols.remove(worst); print("убран признак с неправильным знаком:", worst)
assert all(c in cols for c in MUST), "обязательный признак выпал"
print(f"\nитоговые признаки скор-карты ({len(cols)}):", cols)
print(pd.Series(lr.coef_[0], index=cols).round(3).sort_values(ascending=False).to_string())

после IV-фильтра: 43, после фильтра корреляций: 30
отброшены по корреляции: ['bank_tenure_months', 'delinquency_severity', 'dscr', 'ebitda_margin', 'existing_payment_to_ebitda', 'liabilities_to_ebitda', 'max_overdue_days', 'net_margin', 'new_payment_to_ebitda', 'revenue_growth_last2q', 'revenue_ltm_log', 'revenue_trend', 'roa']
убран признак с неправильным знаком: industry
убран признак с неправильным знаком: payables_days
убран признак с неправильным знаком: debt_to_revenue
убран признак с неправильным знаком: turnover_trend
убран признак с неправильным знаком: neg_balance_months

итоговые признаки скор-карты (25): ['payment_burden', 'loss_quarters', 'equity_ratio', 'negative_equity', 'gross_margin', 'revenue_growth_yoy', 'liabilities_to_revenue', 'balance_to_turnover', 'inquiries_12m', 'overdue_30_cnt', 'inquiries_3m', 'turnover_cv', 'revenue_cv', 'overdue_60_cnt', 'current_overdue_flag', 'current_ratio', 'overdue_90_cnt', 'company_age_months', 'tax_debt_ratio', 'open_loans', 'histor

## 5. Baseline: логистическая регрессия на WOE. Подбор регуляризации на valid, устойчивость по 5-fold CV на train

In [7]:
W_va = woe.transform(Xc.iloc[va])
res = []
for C in (0.01, 0.03, 0.1, 0.3, 1.0, 3.0):
    m = LogisticRegression(C=C, max_iter=2000).fit(W_tr[cols], y[tr])
    res.append((C, M.gini(y[va], m.predict_proba(W_va[cols])[:, 1]), M.brier(y[va], m.predict_proba(W_va[cols])[:, 1])))
print(pd.DataFrame(res, columns=["C", "gini_valid", "brier_valid"]).round(4).to_string(index=False))
best_C = max(res, key=lambda t: t[1])[0]; print("выбрано C =", best_C)

# CV Gini на train: WOE-биннинг пересчитывается внутри каждого фолда (без утечки)
skf = StratifiedKFold(5, shuffle=True, random_state=42); cv_g = []
for a, b in skf.split(tr, y[tr]):
    ia, ib = tr[a], tr[b]
    e = WOEEncoder(num_cols=NUM_FEATURES + BIN_FEATURES, cat_cols=CAT_FEATURES, directions=dirs).fit(Xc.iloc[ia], y[ia])
    mm = LogisticRegression(C=best_C, max_iter=2000).fit(e.transform(Xc.iloc[ia])[cols], y[ia])
    cv_g.append(M.gini(y[ib], mm.predict_proba(e.transform(Xc.iloc[ib])[cols])[:, 1]))
print("CV Gini по фолдам:", np.round(cv_g, 3), "| среднее", round(np.mean(cv_g), 3), "± ", round(np.std(cv_g), 3))

   C  gini_valid  brier_valid
0.01      0.6509       0.0748
0.03      0.6492       0.0749
0.10      0.6434       0.0752
0.30      0.6381       0.0756
1.00      0.6347       0.0759
3.00      0.6334       0.0760
выбрано C = 0.01


CV Gini по фолдам: [0.582 0.593 0.654 0.667 0.659] | среднее 0.631 ±  0.036


In [8]:
def make_woe_pipeline(cols, C):
    return Pipeline([
        ("features", FunctionTransformer(compute_features)),
        ("clip", QuantileClipper()),
        ("woe", WOEEncoder(num_cols=NUM_FEATURES + BIN_FEATURES, cat_cols=CAT_FEATURES, directions=dirs)),
        ("select", ColumnSelector(cols)),
        ("lr", LogisticRegression(C=C, max_iter=2000)),
    ])
pipe_lr = make_woe_pipeline(cols, best_C).fit(raw.iloc[tr], y[tr])
p_va_lr = pipe_lr.predict_proba(raw.iloc[va])[:, 1]
print("baseline WOE-LR, valid:", {k: round(v, 4) for k, v in M.summary(y[va], p_va_lr).items() if k not in ('name',)})

baseline WOE-LR, valid: {'n': 2400, 'default_rate': 0.1025, 'auc': 0.8254, 'gini': 0.6509, 'ks': 0.5457, 'brier': 0.0748, 'ece': 0.0129, 'mean_pd': 0.1031}


## 6. Калибровка: Platt vs isotonic
Выбор метода — 5-fold кросс-валидация **внутри valid** (на test не смотрим). Критерий: Brier и ECE.

In [9]:
from sklearn.isotonic import IsotonicRegression
def fit_cal(kind, p, yy):
    if kind == "platt":
        return LogisticRegression(C=1e6, max_iter=1000).fit(M_logit(p).reshape(-1, 1), yy)
    return IsotonicRegression(out_of_bounds="clip", y_min=0.002, y_max=0.95).fit(p, yy)
def M_logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def apply_cal(kind, c, p): return c.predict_proba(M_logit(p).reshape(-1, 1))[:, 1] if kind == "platt" else c.predict(p)

def cv_cal(p, yy, kind):
    out = np.zeros(len(p))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=1).split(p, yy):
        out[b] = apply_cal(kind, fit_cal(kind, p[a], yy[a]), p[b])
    return np.clip(out, 0.002, 0.95)
rows = []
for nm, p in (("WOE-LR", p_va_lr),):
    rows.append(dict(model=nm, method="без калибровки", brier=M.brier(y[va], p), ece=M.ece(y[va], p)))
    for kind in ("platt", "isotonic"):
        pc = cv_cal(p, y[va], kind)
        rows.append(dict(model=nm, method=kind, brier=M.brier(y[va], pc), ece=M.ece(y[va], pc)))
print(pd.DataFrame(rows).round(5).to_string(index=False))

 model         method   brier     ece
WOE-LR без калибровки 0.07478 0.01290
WOE-LR          platt 0.07487 0.00847
WOE-LR       isotonic 0.07564 0.01751


## 7. Сохранение артефакта

Модель — WOE-логрегрессия, которая напрямую переводится в скор-карту (каждый признак → баллы), что и требует ТЗ («каждому значимому признаку соответствует определённое влияние на итоговую оценку»). Метод калибровки выбираем по Brier на valid; артефакт сохраняем вместе с версией и метаданными обучения.

In [10]:
def pick_cal(p):
    """Platt по умолчанию (гладкий, монотонный); isotonic - только если Brier лучше более чем на 2%."""
    b_platt, b_iso = (M.brier(y[va], cv_cal(p, y[va], k)) for k in ("platt", "isotonic"))
    return "isotonic" if b_iso < 0.98 * b_platt else "platt"
kind_lr = pick_cal(p_va_lr)
print("калибровка:", kind_lr)

VERSION = "v1.0.0"
def build(pipe, p_va, kind, name):
    cal = fit_cal(kind, p_va, y[va])
    return CalibratedPD(pipe, cal, kind, version=f"{VERSION}-{name}",
                        meta=dict(name=name, trained_at=time.strftime("%Y-%m-%d"), dataset_version="synth-v1.0", seed=42,
                                  n_train=int(len(tr)), n_valid=int(len(va)), n_test=int(len(te))))
model_lr = build(pipe_lr, p_va_lr, kind_lr, "woe-logit")
MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(model_lr, MODELS / "risk_model_woe_logit_v1.joblib")
json.dump(dict(main=model_lr.meta["name"], main_version=model_lr.version, scorecard_features=cols, woe_logit_C=best_C,
               calibration=dict(woe_logit=kind_lr), cv_gini_woe_logit=[float(x) for x in cv_g]),
          open(MODELS / "training_meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("сохранено:", sorted(p.name for p in MODELS.iterdir()))
print("модель:", model_lr.version, "| калибровка:", model_lr.method)

калибровка: platt
сохранено: ['rating_scale_v1.json', 'risk_model_woe_logit_v1.joblib', 'scorecard_v1.json', 'training_meta.json']
модель: v1.0.0-woe-logit | калибровка: platt


## 8. Проверка артефакта: загрузка с диска даёт тот же результат (детерминированность)

In [11]:
re = joblib.load(MODELS / "risk_model_woe_logit_v1.joblib")
a, b = model_lr.predict_pd(raw.iloc[va]), re.predict_pd(raw.iloc[va])
print("совпадение прогнозов после сохранения/загрузки:", bool(np.array_equal(a, b)))
print("повторный вызов даёт то же:", bool(np.array_equal(re.predict_pd(raw.iloc[:100]), re.predict_pd(raw.iloc[:100]))))
print("средний PD на valid:", round(a.mean(), 4), "| фактическая доля дефолтов:", round(y[va].mean(), 4))

совпадение прогнозов после сохранения/загрузки: True
повторный вызов даёт то же: True
средний PD на valid: 0.1025 | фактическая доля дефолтов: 0.1025
